In [ ]:
import pandas as pd
import os

In [ ]:
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

%env USER_NAME={USER_NAME}

In [ ]:
cur_dir = '/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta'

# Make parameter file to run parallel jobs

In [ ]:
# Initialize an Empty DataFrame
param = []

my_bucket = os.getenv('WORKSPACE_BUCKET')

for anc in ["eur", "afr", "eas", "amr", "oth", "mid", "sas"]:
    param.append({'--input-recursive plinkpath' : f"{my_bucket}/data/meta/plink_array/qc_{anc}",
                  '--output-recursive outpath' : f"{my_bucket}/data/meta/pca/pca_{anc}",
                  '--env ancestry' : anc})

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME = f'{cur_dir}/pca_bygroup.tsv'

param.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

%env PARAMETER_FILENAME={PARAMETER_FILENAME}

# PCA for EUR

The parameter file is same

## Script to compute sumstats

Since this is a dsub job, we need the input files to be copied to the container and the input files out of the container. So they are specified within the dsub jobs

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script takes in the QC'd link array data and
# performs PCA on it using PCair controlled for relatedness

set -o errexit
set -o nounset

plink2 \
    --bfile "${plinkpath}/plinkarray_qc_${ancestry}" \
    --exclude "${plinkpath}/plinkarray_qc_${ancestry}.prune.out" \
    --memory 290000 \
    --pca 20 approx \
    --out "${outpath}/${ancestry}"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_EUR_only.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_EUR_only.sh

## Submit dsub job

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image biocontainer/plink2:alpha2.3_jan2020 \
    --min-ram 300 \
    --min-cores 15 \
    --disk-size 300 \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}.log" \
    --tasks "${PARAMETER_FILENAME}" 1 \
    --preemptible \
    --retries 2 \
    --wait \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_EUR_only.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

## Read log files

In [ ]:
!gsutil cat gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/meta/pca/pca_eur/eur.log

#  PCA for non-EUR subgroups

## Script to compute sumstats

Since this is a dsub job, we need the input files to be copied to the container and the input files out of the container. So they are specified within the dsub jobs

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script takes in the QC'd link array data and
# performs PCA on it using PCair controlled for relatedness

set -o errexit
set -o nounset

Rscript --vanilla \
    "${PCASCRIPT}" \
    --plinkfile "${plinkpath}/plinkarray_qc_${ancestry}" \
    --savefile "${outpath}/${ancestry}"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_bygroup.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_bygroup.sh

## Submit dsub job

This script makes a nice folder structure within GCP and does the job over there. The next cell outputs a python env variable called job_id

In [ ]:
#!gsutil rm -r ${WORKSPACE_BUCKET}/data/logs/pca-bygroup

For Africans, the whole job takes 10 hours

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image abiji/aou_step0:v2 \
    --machine-type "m1-megamem-96" \
    --disk-size 2000 \
    --input PCASCRIPT="${WORKSPACE_BUCKET}/data/scripts/r/pca_by_inferred_ancestry.R" \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}.log" \
    --tasks "${PARAMETER_FILENAME}" 2 \
    --preemptible \
    --retries 1 \
    --wait \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_bygroup.sh"

AMR is the most time consuming, and it takes 7-8 hours

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image abiji/aou_step0:v2 \
    --min-ram 500 \
    --min-cores 26 \
    --disk-size 1500 \
    --input PCASCRIPT="${WORKSPACE_BUCKET}/data/scripts/r/pca_by_inferred_ancestry.R" \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}.log" \
    --tasks "${PARAMETER_FILENAME}" 3-7 \
    --preemptible \
    --retries 1 \
    --wait \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/pca/pca_bygroup.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

## Read log files

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/logs/pca-bygroup/pca-bygrou--bijia01--240825-050939-20-2.log